# Feature Engineering

## Preliminaries

In [ ]:
# Load libraries
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Load data
train = pd.read_csv('../data/raw/train.csv', sep='|')
items = pd.read_csv('../data/raw/items.csv', sep='|')

df = train.merge(items, 
                   on='pid', 
                   how='left',
                   validate="m:1")

## Clean Data

### Handling Missing Values

Handling missing values is within the Data Prepration step according to the CRISP-DM process. It's about cleaning data, which often this is the lengthiest task. Without it, you’ll likely fall victim to garbage-in, garbage-out. A common practice during this task is to correct, impute, or remove erroneous values.

Our EDA has identified missing values in category, campaignIndex, pharmForm, and competitorPrice. We need to decide on a strategy for each:
- Imputation: Fill missing values with a statistic like the mean, median, or mode.
- Removal: Remove rows with missing values (use with caution, as it can lead to data loss).
- Model-based imputation: Use a model to predict the missing values.
- Create a new category: For categorical variables, you could treat "missing" as a new category.

In [ ]:
# Analyse missing values to determine how to handle them in the next steps. sort by missing percentage in descending order
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing_values, 'missing_percentage': missing_percentage})
missing_df = missing_df.sort_values(by='missing_percentage', ascending=False)
print(missing_df)

                 missing_count  missing_percentage
campaignIndex          2287968               83.02
pharmForm               194124                7.04
competitorPrice         100687                3.65
category                 87394                3.17
lineID                       0                0.00
availability                 0                0.00
adFlag                       0                0.00
pid                          0                0.00
day                          0                0.00
price                        0                0.00
click                        0                0.00
basket                       0                0.00
order                        0                0.00
group                        0                0.00
manufacturer                 0                0.00
revenue                      0                0.00
content                      0                0.00
genericProduct               0                0.00
unit                         0 

In [ ]:
# Analyse if the absence of a value in each column might be correlated with whether an order is placed or not incl. percentage
missing_order_correlation = {}
for column in df.columns:
    missing_order_correlation[column] = df.groupby(df[column].isnull())['order'].mean().round(4) * 100
missing_order_correlation_df = pd.DataFrame(missing_order_correlation).T
missing_order_correlation_df.columns = ['order_percentage_when_missing', 'order_percentage_when_present']
missing_order_correlation_df = missing_order_correlation_df.sort_values(by='order_percentage_when_missing', ascending=False)
print(missing_order_correlation_df)

                 order_percentage_when_missing  order_percentage_when_present
campaignIndex                            27.45                          25.20
pharmForm                                26.02                          19.79
competitorPrice                          25.94                          16.17
category                                 25.83                          18.06
lineID                                   25.58                            NaN
availability                             25.58                            NaN
adFlag                                   25.58                            NaN
pid                                      25.58                            NaN
day                                      25.58                            NaN
price                                    25.58                            NaN
click                                    25.58                            NaN
basket                                   25.58                  

### campainIndex

- MNAR (Missing Not at Random): With an 83% missing rate, the absence of a value is the dominant feature. The order rate differs depending on whether a value is present or not. This indicates that the missingness is not random. It's directly related to the unobserved value itself—specifically, the product was likely not part of a campaign, hence no campaignIndex was recorded. The reason for missingness is the value itself (or lack thereof), which is the definition of MNAR. Our strategy to drop the column is justified because the feature provides very little information beyond its own absence.

With 83.02% of its values missing, the campaignIndex column is mostly empty. This significantly limits its usefulness. Standard imputation methods like using the mean, median, or mode are not suitable here, as filling over 80% of the data with a single value would heavily distort the feature's distribution and likely mislead the model.

The crosstabulation shows a clear difference in ordering behavior depending on whether campaignIndex is missing.
- When campaignIndex has a value: The order rate is approximately 27.45% (128,467 orders / 468,035 total).
- When campaignIndex is missing: The order rate is approximately 25.19% (576,623 orders / 2,287,968 total).

While there is a slight difference, the fact that the column is overwhelmingly empty suggests that its predictive power is likely low. The information it provides is minimal because for the vast majority of the data, the only signal is "the value is missing."

Given the extremely high percentage of missing data, the best course of action is to drop the campaignIndex column entirely.

Reasoning:
1. Low Information Content: A feature that is empty 83% of the time is unlikely to provide significant predictive value to a model.
2. Risk of Noise: Keeping the column, even by creating a "missing" category, could introduce noise and complexity to the model for very little gain.
3. Simplification: Removing the column simplifies the feature set, which can lead to a more robust and interpretable model.

In [4]:
# Drop the 'campaignIndex' column from the DataFrame
df.drop(columns=['campaignIndex'], inplace=True)

# Checking remaining columns
print(df.columns)

Index(['lineID', 'day', 'pid', 'adFlag', 'availability', 'competitorPrice',
       'click', 'basket', 'order', 'price', 'revenue', 'manufacturer', 'group',
       'content', 'unit', 'pharmForm', 'genericProduct', 'salesIndex',
       'category', 'rrp'],
      dtype='str')


### pharmForm

- MNAR (Missing Not at Random): The order rate is significantly higher (26.02%) when pharmForm is missing compared to when it is present (19.79%). This strong correlation with the target variable rules out MCAR and MAR. The missingness is systematic and related to the nature of the product itself. As we hypothesized, certain types of products (like cosmetics or devices) may not have a standard pharmaceutical form, and these same products have a higher propensity to be ordered. The missingness is part of the feature's story, making it MNAR. Our strategy to create a 'Missing' category correctly treats this as an informative signal.

The `pharmForm` column has a missing value rate of 7.04%, which is a moderate but significant portion of the data. Simply dropping these rows could discard valuable information and potentially introduce bias if the missingness is not random.

The provided crosstabulation against the `order` variable is highly revealing:
-   **Order Rate when `pharmForm` is Missing**: 26.02%
-   **Order Rate when `pharmForm` is Present**: 19.79%

This disparity indicates that the missingness in `pharmForm` is **not random**; it is informative. Products with a missing `pharmForm` have a noticeably higher propensity to be ordered. This suggests that the absence of this feature is a predictive signal in itself.

There could be several underlying business reasons for this pattern:
1.  **Product Type**: Certain types of products that sell well might not have a standard pharmaceutical form (e.g., cosmetics, health devices, or bundled items).
2.  **Data Entry Practices**: It's possible that for certain popular or fast-moving items, data entry for this specific field is deprioritized, leading to missing values.
3.  **Customer Perception**: The absence of a specific pharmaceutical form might make a product appear more like a general consumer good rather than a specialized medical item, which could broaden its appeal and increase sales.

**Recommended Strategy: Imputation with a 'Missing' Category**

Given that the missingness is a predictive signal, the best approach is to preserve this information. I recommend imputing the `NaN` values with a distinct string, such as `'Missing'`.

**Justification:**
-   **Preserves Information**: This method explicitly captures the signal that the absence of `pharmForm` provides, allowing the machine learning model to learn the relationship between "missingness" and the higher order rate.
-   **Avoids Data Loss**: It prevents the loss of 7.04% of the rows in your dataset, which would happen if you chose to drop them.
-   **Prevents Distortion**: Unlike mean/median/mode imputation, creating a new category does not distort the distribution of existing `pharmForm` categories. It treats the missing data as a unique group, which, according to the data, it is.

This strategy effectively turns the missing data into a useful feature for the model.

In [ ]:
# Impute missing values in 'pharmForm' with the string 'Missing'
df['pharmForm'] = df['pharmForm'].fillna('Missing')
print(df['pharmForm'].isnull().sum())

0


### competitorPrice

- MNAR (Missing Not at Random): This is another clear case of MNAR. The order rate is dramatically higher (25.94%) when competitorPrice is missing. The most logical explanation is that the missingness is due to the product being exclusive, a unique offering for which no competitor price exists. The reason for the missing value (exclusivity) is directly related to the value that is missing and has a strong influence on the outcome. Our two-step strategy of creating a binary flag (competitorPrice_is_missing) and then imputing the original column was designed specifically to handle this type of informative MNAR data.

The `competitorPrice` column has a relatively low missing value rate of 3.65%. While this percentage is small, the relationship between its missingness and the `order` rate is too significant to ignore.

The crosstabulation reveals a stark contrast:
-   **Order Rate when `competitorPrice` is Missing**: 25.94%
-   **Order Rate when `competitorPrice` is Present**: 16.17%

This is a substantial difference, indicating that products without a listed competitor price are ordered much more frequently. The missingness is clearly **informative** and not random. Dropping these rows would mean discarding a strong predictive signal.

Several business scenarios could explain this phenomenon:
1.  **Exclusive or Unique Products**: The most likely reason is that these products are exclusive to your store. If there are no direct competitors, there is no competitor price to list. Such exclusive items are often highly sought after, leading to a higher purchase rate.
2.  **Loss-Leader or "Too Low to Show" Pricing**: The product's price might be so competitive that the business has chosen not to display the competitor's price to avoid direct price wars or to comply with pricing agreements. This aggressive pricing would naturally drive more orders.
3.  **New Product Launches**: For newly introduced items, the data collection process for competitor pricing may not have been completed yet. These new products might have high initial sales due to launch promotions or novelty effects.

We considered using a product's own price or rrp to fill the missing competitorPrice values. While this seems intuitive, we decided against it for a crucial reason: it would fundamentally alter the meaning of the feature.
The competitorPrice column represents external market data, whereas our price and rrp are internal strategic values. By using our own price for imputation, we would be erasing the powerful signal that the data is missing, which often indicates an exclusive product with no direct competitor. This could mislead the model by creating an artificial relationship in the data.
Therefore, we concluded that creating a separate binary flag to capture the "missingness" and then using a neutral value like the median for imputation is a more robust approach that preserves the feature's integrity.

Instead, because `competitorPrice` is a numerical feature, simply filling it with a value like -1 or 0 could be misinterpreted by some models as a very low price. A more robust approach is to explicitly capture the "missingness" as a feature and then impute the missing price values in a way that doesn't distort the original distribution.

We choose a two-step strategy:
1.  **Create a Binary Flag**: Create a new column, for example `competitorPrice_is_missing`, that is `1` if the `competitorPrice` was missing and `0` otherwise. This directly feeds the powerful "missingness" signal into the model.
2.  **Impute the Original Column**: After creating the flag, you can impute the missing values in the original `competitorPrice` column. A good choice would be the **median** competitor price. Using the median is generally more robust to outliers than the mean. This step ensures the column remains a valid numerical feature without `NaN` values.

**Justification:**
-   **Captures the Signal**: The binary flag ensures the model can directly learn the strong relationship between the absence of a competitor price and a higher order rate.
-   **Preserves the Feature**: Imputing the original column allows you to still use `competitorPrice` as a numerical feature (e.g., to calculate price differences), without having to deal with `NaN`s.
-   **Robustness**: This two-part strategy is more robust and informative than simply filling the missing values with a placeholder like 0 or -1, which could be misinterpreted by the model.

In [ ]:
# Create a binary flag for missing competitorPrice
df['competitorPrice_is_missing'] = df['competitorPrice'].isnull().astype(int)

# Calculate the median of the 'competitorPrice' column
median_competitor_price = df['competitorPrice'].median()

# Impute missing values in 'competitorPrice' with the median
df['competitorPrice'] = df['competitorPrice'].fillna(median_competitor_price)

print(f"Median competitor price used for imputation: {median_competitor_price}")
df[['competitorPrice', 'competitorPrice_is_missing']].head()

### category

- Likely MNAR (Missing Not at Random), possibly MAR: Similar to the other features, the order rate is higher when the category is missing. This suggests the missingness is not completely random (ruling out MCAR). It is likely MNAR for the same reasons as pharmForm—the absence of a category may be a characteristic of certain types of products that sell differently. It could also be considered MAR (Missing at Random) if we argued that the missingness could be predicted from other features in the dataset (e.g., perhaps products from a certain manufacturer are more likely to have missing categories). However, given the direct link to ordering behavior, treating it as MNAR is the safer and more robust assumption. Our strategy to create a 'Missing' category is effective for both MAR and MNAR scenarios where the missingness is informative.

Based on the EDA analysis, the `category` column has a high number of unique values and a long-tail distribution. The code also calculates the percentage of missing values for each column.
As the percentage of missing `category` values is small, we could consider imputing them with the mode (most frequent category). However, given its high cardinality, a better approach is to treat "missing" as a separate category. This can even be a useful predictive signal, as the absence of a category might be correlated with whether an order is placed.

Why this is the best strategy:
1. Preserves Data: 3.17% is a significant amount of data to simply delete. Removing these rows could eliminate valuable information from other columns.
2. Avoids Skewing the Distribution: The category feature has high cardinality (many unique values) and a long-tail distribution. Imputing with the most frequent category (the mode) would artificially inflate its importance and might mislead the model.
3. Potential Predictive Power: The fact that a category is missing might itself be a useful signal for the model. For example, products without a category might have a different ordering behavior. Creating a separate "Missing" or "Unknown" category allows the model to learn this potential relationship.

In [ ]:
# Impute missing values in 'category' with 'Missing'
df['category'] = df['category'].fillna('Missing')

## Construct Data

Construct data: Derive new attributes that will be helpful. For example, derive someone’s body mass index from height and weight fields.